```yaml
title: Building an incumbent radar from one UEI
description: Pull a competitor's contracts, IDVs, OTAs, and subaward flow from one UEI to surface agency mix, NAICS mix, and recompete windows.
tags: [entities, contracts, joining, capture]
endpoints: [list_entities, get_entity, list_entity_contracts, list_entity_idvs, list_entity_otas, list_entity_subawards]
```

# Building an incumbent radar from one UEI

Capture starts with a question that's annoyingly hard to answer from public sources: *what does this competitor actually do for the government, and where are their contracts about to expire?* The Tango entity endpoints answer it in five calls — comprehensive entity profile, then contracts, IDVs, OTAs, and subawards scoped to the same UEI.

## Setup

In [1]:
from collections import Counter, defaultdict
from datetime import date
from tango import TangoClient

client = TangoClient()

## Resolve a name to a UEI

You usually start with a company name, not a UEI. `list_entities(name=...)` is the right primary lookup — it searches the entity registry and gives you the legal business name to confirm you've grabbed the right corporate shell (Leidos has plenty of siblings).

In [2]:
TARGET = "JOHNSON CONTROLS BUILDING AUTOMATION SYSTEMS, LLC"

candidates = client.list_entities(name=TARGET, limit=5)
for e in candidates.results:
    print(f"  {e.get('uei')}  {e.get('legal_business_name')}")

# Pick the top hit. In real workflows, confirm against CAGE / DBA / address.
UEI = candidates.results[0]["uei"]
print(f"\nUsing UEI={UEI}")

  LFD2YFMZJ1J3  JOHNSON CONTROLS BUILDING AUTOMATION SYSTEMS, LLC
  G15SWMLGAX81  JOHNSON CONTROLS BUILDING AUTOMATION SYSTEMS, LLC

Using UEI=LFD2YFMZJ1J3


## The entity profile

`get_entity` returns the comprehensive shape by default — capabilities text, business types, NAICS/PSC mix, and federal obligations rollups. It's the one call that gives you a one-pager.

In [3]:
entity = client.get_entity(UEI)

print(f"{entity.get('legal_business_name')}  ({entity.get('cage_code')})")
print(f"Primary NAICS: {entity.get('primary_naics')}")
naics = entity.get("naics_codes") or []
print(f"Registered NAICS count: {len(naics)}")
psc = entity.get("psc_codes") or []
print(f"Registered PSC count:   {len(psc)}")
fo = entity.get("federal_obligations") or {}
if fo:
    print(f"\nFederal obligations rollup (from entity profile):")
    for k, v in fo.items():
        print(f"  {k}: {v}")

JOHNSON CONTROLS BUILDING AUTOMATION SYSTEMS, LLC  (4CWR3)
Primary NAICS: 236220
Registered NAICS count: 26
Registered PSC count:   7

Federal obligations rollup (from entity profile):
  total: {'idv_count': 16, 'awards_count': 662, 'subawards_count': 23, 'awards_obligated': 2159651162.14, 'subawards_obligated': 2442758.62}
  active: {'idv_count': 1, 'awards_count': 73, 'awards_obligated': 527629645.09}


## Contracts: which agencies pay this competitor

`list_entity_contracts` is the same filter set as `list_contracts`, scoped to a single recipient. Shape it to pull just the awarding office and dollar fields — that's all you need for an agency rollup.

In [4]:
CONTRACTS_SHAPE = (
    "key,piid,award_date,fiscal_year,obligated,total_contract_value,"
    "naics_code,psc_code,"
    "awarding_office(*)"
)
MAX_PAGES = 5  # ~500 contracts is plenty for an agency profile

contracts, cursor = [], None
for _ in range(MAX_PAGES):
    r = client.list_entity_contracts(
        UEI, fiscal_year_gte=2022, shape=CONTRACTS_SHAPE, limit=100, cursor=cursor,
    )
    contracts.extend(r.results)
    cursor = r.cursor
    if not cursor:
        break

print(f"Pulled {len(contracts)} contracts since FY22.")

by_dept = defaultdict(float)
for c in contracts:
    dept = (c.get("awarding_office") or {}).get("department_name") or "(unknown)"
    by_dept[dept] += float(c.get("obligated") or 0)

print("\nTop departments by obligated dollars (FY22+):")
for dept, dollars in sorted(by_dept.items(), key=lambda x: -x[1])[:8]:
    print(f"  ${dollars/1e6:>8.1f}M  {dept}")

Pulled 98 contracts since FY22.

Top departments by obligated dollars (FY22+):
  $   500.9M  DEPT OF DEFENSE
  $     0.2M  STATE, DEPARTMENT OF
  $     0.1M  HEALTH AND HUMAN SERVICES, DEPARTMENT OF


## NAICS and PSC mix

Where do this company's contracts cluster? NAICS tells you the industry framing; PSC tells you what's actually being bought. A wide spread says diversified prime; a narrow one says specialist.

In [5]:
naics_dollars = defaultdict(float)
psc_dollars = defaultdict(float)
for c in contracts:
    obl = float(c.get("obligated") or 0)
    naics_dollars[str(c.get("naics_code") or "")] += obl
    psc_dollars[str(c.get("psc_code") or "")] += obl

print("Top NAICS by obligated:")
for n, d in sorted(naics_dollars.items(), key=lambda x: -x[1])[:6]:
    print(f"  ${d/1e6:>8.1f}M  NAICS {n}")

print("\nTop PSC by obligated:")
for p, d in sorted(psc_dollars.items(), key=lambda x: -x[1])[:6]:
    print(f"  ${d/1e6:>8.1f}M  PSC {p}")

Top NAICS by obligated:
  $   498.2M  NAICS 541512
  $     2.8M  NAICS 561621
  $     0.1M  NAICS 334118
  $     0.1M  NAICS 238210
  $     0.1M  NAICS 333415
  $     0.0M  NAICS 221310

Top PSC by obligated:
  $   390.4M  PSC N059
  $   102.8M  PSC J059
  $     5.0M  PSC Z1AZ
  $     2.8M  PSC J063
  $     0.1M  PSC 7A20
  $     0.1M  PSC Z1PZ


## IDVs: where the recompete clock is ticking

Task-order contracts roll under IDVs (GWACs, MACs, agency BPAs). The IDV's `period_of_performance.last_date_to_order` is the date orders stop flowing — that's the meaningful recompete signal. Pull the IDVs and sort by expiry.

In [6]:
IDV_SHAPE = (
    "key,piid,award_date,idv_type,description,total_contract_value,obligated,"
    "awarding_office(agency_name,department_name),"
    "period_of_performance(start_date,last_date_to_order)"
)

idvs, cursor = [], None
for _ in range(3):
    r = client.list_entity_idvs(UEI, shape=IDV_SHAPE, limit=100, cursor=cursor)
    idvs.extend(r.results)
    cursor = r.cursor
    if not cursor:
        break

today = date.today()
active = [
    i for i in idvs
    if i["period_of_performance"]["last_date_to_order"] and i["period_of_performance"]["last_date_to_order"] >= today
]
active.sort(key=lambda i: (i.get("period_of_performance") or {}).get("last_date_to_order") or "9999")

print(f"{len(idvs)} IDVs total, {len(active)} still on the clock.\n")
print(f"  {'Expires':<12} {'Obligated':>10}  Dept / piid / one-liner")
for i in active[:10]:
    # last_date_to_order is a date object; wrap in str() so f-string column padding
    # doesn't get interpreted as a strftime spec (which would silently print "<12").
    expiry = str((i.get("period_of_performance") or {}).get("last_date_to_order") or "?")
    dept = (i.get("awarding_office") or {}).get("department_name") or "?"
    obl = float(i.get("obligated") or 0) / 1e6
    desc = (i.get("description") or "").replace("\n", " ")[:60]
    print(f"  {expiry:<12} ${obl:>8.1f}M  {dept[:18]:<18}  {i.get('piid','?')}  {desc}")

16 IDVs total, 1 still on the clock.

  Expires       Obligated  Dept / piid / one-liner
  2029-08-19   $     0.0M  DEPT OF DEFENSE     W9127824D0058  DESIGN BUILD CONSTRUCTION IDIQ MATOC - PACIFIC REGION


## OTAs and subawards: the off-paper picture

Other Transaction Authorities (OTAs) sit outside the FAR and are where DoD and HHS are doing their most flexible buying. `list_entity_subawards` returns subawards this UEI shows up *on* — usually as the subawardee — so aggregating by `prime_recipient` tells you which primes carry this competitor on their bench. That's BD intel in both directions: if you're trying to displace them, these are the primes you have to peel them off of.

In [7]:
otas = client.list_entity_otas(UEI, limit=50).results
subs = client.list_entity_subawards(UEI, limit=50).results

print(f"OTAs:      {len(otas)} on file (showing first page)")
for o in otas[:5]:
    rec = (o.get("recipient") or {}).get("display_name") or "?"
    desc = (o.get("description") or "").replace("\n", " ")[:70]
    print(f"  {o.get('piid','?')}  {rec[:30]}  {desc}")

print(f"\nSubawards: {len(subs)} on file (showing first page)")
prime_payers = Counter(
    (s.get("prime_recipient") or {}).get("display_name") for s in subs
)
print("Primes that pay this UEI as a sub:")
for name, count in prime_payers.most_common(8):
    if name:
        print(f"  {count:>3}x  {name}")

OTAs:      0 on file (showing first page)

Subawards: 23 on file (showing first page)
Primes that pay this UEI as a sub:
   18x  BAE SYSTEMS HAWAII SHIPYARDS INC.
    2x  BAE SYSTEMS MARITIME SOLUTIONS NORFOLK INC.
    1x  BROOKS RANGE CONTRACT SERVICES, INC.
    1x  BAE SYSTEMS MARITIME SOLUTIONS SAN DIEGO INC.
    1x  HII FLEET SUPPORT GROUP LLC
